# 09_ImageFeatures — Trích đặc trưng hình ảnh từ video gốc (Task 7a)

Bước đầu của việc so sánh model hình ảnh với model landmark. Notebook này **không train gì**, chỉ biến mỗi video gốc thành một file đặc trưng.

**Làm gì:** với mỗi video, đọc **mọi khung hình** (60 hoặc 120 fps, không bỏ khung nào), thu nhỏ cả khung về 224×224, cho qua ResNet18 đã học sẵn (bỏ lớp phân loại) để ra 512 số cho mỗi khung. Mỗi video có **4 phiên bản**: gốc, lật gương, augmentation (cắt phóng nhẹ + đổi sáng/tương phản), augmentation + lật gương. Một phiên bản dùng chung một phép biến đổi cho mọi khung của video.

**Lưu ở:** `Data_321/img_feats/` — mỗi video một file `.npz` (đặc trưng `f` kích thước `(4, số khung, 512)` và `fps` của video gốc), cùng bảng `video_map.csv` ghép dòng manifest với video gốc. Chạy lại sẽ bỏ qua file đã có.

**Cách chạy:** kích hoạt môi trường `C:\envs\train-gpu\Scripts\activate.bat`, mở notebook bằng kernel `train-gpu`, chạy lần lượt cell 1 → 6. Lần đầu chạy cell 3 sẽ tải trọng số ResNet18 nên cần internet.

**Lưu ý:** cell 2 ghép video theo **tên file** và kiểm tra lại bằng số khung. Nếu không ghép được, nó dừng và in ví dụ để bạn gửi cho mình chỉnh.

In [1]:
#cell 1: thư viện, kiểm tra CUDA, đọc manifest
import time
import numpy as np, pandas as pd, torch, cv2
import torch.nn as nn, torch.nn.functional as F
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook này không chạy trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
FEAT_DIR = BASE / "img_feats"
FEAT_DIR.mkdir(exist_ok=True)
CLASSES = ["paper", "rock", "scissors"]
assert (BASE / "manifest.csv").exists(), f"Không thấy {BASE / 'manifest.csv'}"
m = pd.read_csv(BASE / "manifest.csv")
print("GPU:", torch.cuda.get_device_name(0))
print("Cột manifest:", list(m.columns))
print("Số video theo tập:", m.tap.value_counts().to_dict())
display(m.head(3))

GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Cột manifest: ['path', 'nguon', 'lop', 'nhan', 'khung', 'fps', 'tap']
Số video theo tập: {'train': 151, 'test': 33, 'val': 31}


,path,nguon,lop,nhan,khung,fps,tap
0,landmarks_120\Dataset_hand\paper\VID_202610020...,Dataset_hand,paper,0,135,120.0,train
1,landmarks_120\Dataset_hand\paper\VID_202610020...,Dataset_hand,paper,0,125,120.0,test
2,landmarks_120\Dataset_hand\paper\VID_202610020...,Dataset_hand,paper,0,121,120.0,val


In [2]:
#cell 2: ghép từng dòng manifest với video gốc (theo tên file), kiểm tra bằng số khung, lưu video_map.csv
VMAP = FEAT_DIR / "video_map.csv"
EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"}

def build_map():
    vids = [p for p in BASE.rglob("*") if p.suffix.lower() in EXTS and FEAT_DIR not in p.parents]
    index = {}
    for p in vids:
        index.setdefault(p.stem, []).append(p)
    print(f"Tìm thấy {len(vids)} video trong {BASE}")

    found, bad = [], []
    for i, r in m.iterrows():
        cands = index.get(Path(r.path).stem, [])
        if len(cands) > 1:                                    # trùng tên: giữ video nằm trong thư mục đúng lớp
            cands = [p for p in cands if CLASSES[int(r.nhan)] in str(p).lower()]
        if len(cands) == 1:
            found.append(str(cands[0]))
        else:
            bad.append((r.path, len(cands)))
    print(f"Ghép được {len(found)}/{len(m)} dòng")
    if bad:
        print("Chưa ghép được (đường dẫn landmark, số video khớp):", bad[:5])
        print("Ví dụ đường dẫn landmark:", list(m.path[:3]))
        print("Ví dụ video tìm thấy:", [str(p) for p in vids[:3]])
    assert not bad, "Có dòng manifest chưa ghép được video: gửi phần in ở trên cho mình để chỉnh cách ghép."

    vm = m.copy()
    vm["video"] = found
    vm["feat"] = [f"{i:04d}_{Path(r.path).stem}.npz" for i, r in vm.iterrows()]
    info = []
    for p in vm.video:                                        # fps, số khung, kích thước của video gốc
        cap = cv2.VideoCapture(p)
        info.append((cap.get(cv2.CAP_PROP_FPS), cap.get(cv2.CAP_PROP_FRAME_COUNT), cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
        cap.release()
    vm[["fps", "n_src", "w", "h"]] = np.array(info)
    assert (vm.fps > 0).all(), "Có video không đọc được fps (file hỏng hoặc thiếu codec)."
    vm["lech"] = (vm.khung - vm.n_src * 120 / vm.fps).abs()   # landmark đã ở 120 fps nên số khung phải khớp
    print("Số video theo (fps, rộng, cao):")
    print(vm.groupby(["fps", "w", "h"]).size())
    n_lech = int((vm.lech > 3).sum())
    print(f"Số khung lệch quá 3 ở {n_lech}/{len(vm)} video")
    assert n_lech < 0.1 * len(vm), "Số khung video và landmark lệch nhiều: có thể ghép nhầm video, gửi phần in ở trên cho mình."
    return vm

vm = pd.read_csv(VMAP) if VMAP.exists() else None
if vm is None or len(vm) != len(m) or not all(Path(p).exists() for p in vm.video):
    vm = build_map()
    vm.to_csv(VMAP, index=False)
    print("Đã lưu", VMAP)
else:
    print("Nạp video_map.csv có sẵn:", len(vm), "video")

Tìm thấy 215 video trong C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321
Ghép được 215/215 dòng
Số video theo (fps, rộng, cao):
fps         w       h     
60.000018   1920.0  1080.0      1
                    1080.0     14
60.000019   1920.0  1080.0     17
60.000029   1920.0  1080.0      1
60.000038   1920.0  1080.0     12
60.000039   1920.0  1080.0     12
60.000055   1920.0  1080.0     13
60.000057   1920.0  1080.0     11
119.945000  1920.0  1080.0     15
120.000000  1920.0  1080.0    119
dtype: int64
Số khung lệch quá 3 ở 0/215 video
Đã lưu C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\img_feats\video_map.csv


In [3]:
#cell 3: CNN nhỏ đã học sẵn (ResNet18, bỏ lớp phân loại) làm bộ trích đặc trưng, chạy CUDA fp16
import torchvision

backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
backbone.fc = nn.Identity()
backbone = backbone.to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    print("Số đặc trưng mỗi khung:", backbone(torch.zeros(2, 3, 224, 224, device=dev)).shape[1])

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Acer Nitro 5 Tiger/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100.0%


Số đặc trưng mỗi khung: 512


In [4]:
#cell 4: đọc video, tạo 4 phiên bản (gốc / lật / augmentation / augmentation + lật), trích đặc trưng
def read_frames(path):
    cap = cv2.VideoCapture(str(path)); frames = []
    while True:
        ok, f = cap.read()
        if not ok:
            break
        frames.append(cv2.resize(cv2.cvtColor(f, cv2.COLOR_BGR2RGB), (224, 224), interpolation=cv2.INTER_AREA))
    cap.release()
    return np.stack(frames)                                   # (T,224,224,3) uint8

def make_version(x, flip, s, dx, dy, b, c):
    """x: (n,3,224,224) float 0-1 trên GPU. Mọi khung của một video dùng chung một phép biến đổi."""
    if s < 1:                                                 # cắt vùng s*224 rồi phóng lại 224
        k = int(224 * s); ox, oy = int(dx * (224 - k)), int(dy * (224 - k))
        x = F.interpolate(x[:, :, oy:oy + k, ox:ox + k], size=224, mode="bilinear", align_corners=False)
    x = ((x - 0.5) * c + 0.5) * b                             # tương phản c, độ sáng b
    return (x.flip(-1) if flip else x).clamp(0, 1)

VERSIONS = [(False, False), (True, False), (False, True), (True, True)]   # (lật gương, augmentation)

@torch.no_grad()
def extract(path, seed):
    x = torch.from_numpy(read_frames(path)).to(dev).permute(0, 3, 1, 2)   # uint8 trên GPU
    rng = np.random.default_rng(seed)
    out = []
    for flip, aug in VERSIONS:
        p = (rng.uniform(.75, 1), rng.random(), rng.random(), *rng.uniform(.7, 1.3, 2)) if aug else (1, 0, 0, 1, 1)
        feats = []
        for i in range(0, len(x), 256):                       # chia lô để không tràn VRAM 4 GB
            xc = make_version(x[i:i + 256].float() / 255, flip, *p)
            with torch.autocast("cuda", dtype=torch.float16):
                feats.append(backbone((xc - MEAN) / STD).float())
        out.append(torch.cat(feats).half().cpu().numpy())
    return np.stack(out)                                      # (4, T, 512) float16

z = extract(vm.video[0], seed=0)
print("Thử video đầu tiên:", z.shape, z.dtype, "| hữu hạn:", bool(np.isfinite(z).all()))

Thử video đầu tiên: (4, 68, 512) float16 | hữu hạn: True


In [5]:
#cell 5: trích đặc trưng cho mọi video (bỏ qua video đã có file), mỗi video một file .npz
t0, done = time.time(), 0
for i, r in vm.iterrows():
    out = FEAT_DIR / r.feat
    if out.exists():
        continue
    tmp = FEAT_DIR / (r.feat[:-4] + ".tmp.npz")               # ghi file tạm rồi đổi tên để file dở dang không bị tính là xong
    np.savez(tmp, f=extract(r.video, seed=1000 + i), fps=r.fps)
    tmp.replace(out)
    done += 1
    if done % 20 == 0:
        print(f"{done} video mới | vị trí {i + 1}/{len(vm)} | {time.time() - t0:.0f}s")
print(f"Xong: {done} video mới trong {time.time() - t0:.0f}s | tổng {len(vm)} video")

20 video mới | vị trí 20/215 | 24s
40 video mới | vị trí 40/215 | 47s
60 video mới | vị trí 60/215 | 70s
80 video mới | vị trí 80/215 | 91s
100 video mới | vị trí 100/215 | 142s
120 video mới | vị trí 120/215 | 175s
140 video mới | vị trí 140/215 | 215s
160 video mới | vị trí 160/215 | 259s
180 video mới | vị trí 180/215 | 308s
200 video mới | vị trí 200/215 | 368s
Xong: 215 video mới trong 398s | tổng 215 video


In [6]:
#cell 6: kiểm tra kết quả — đủ file, không có NaN, số khung khớp video gốc, dung lượng
miss = [f for f in vm.feat if not (FEAT_DIR / f).exists()]
print("Thiếu file:", len(miss))
assert not miss, "Chạy lại cell 5 để trích nốt các video còn thiếu."

nonfinite, diff, size = 0, [], 0
for f, n_src in zip(vm.feat, vm.n_src):
    z = np.load(FEAT_DIR / f)["f"]
    nonfinite += int(not np.isfinite(z).all())
    diff.append(abs(z.shape[1] - n_src))                      # số khung đọc được so với thuộc tính của video
    size += (FEAT_DIR / f).stat().st_size
print(f"File có giá trị NaN/inf: {nonfinite} | lệch số khung tối đa so với thuộc tính video: {max(diff):.0f}")
print(f"Tổng dung lượng img_feats: {size / 1e6:.0f} MB | ví dụ: {np.load(FEAT_DIR / vm.feat[0])['f'].shape}")

Thiếu file: 0
File có giá trị NaN/inf: 0 | lệch số khung tối đa so với thuộc tính video: 0
Tổng dung lượng img_feats: 90 MB | ví dụ: (4, 68, 512)
